# Task #57 — Thêm đặc trưng khoảng cách người bán↔khách hàng (Story #10)

Bảng `geolocation` chưa được join ở Story #4 do ~26% dòng trùng lặp theo `zip_code_prefix` (nhiều toạ độ GPS khác nhau cho cùng 1 prefix). Notebook này: gộp `geolocation` về 1 dòng/prefix bằng **median** lat/lng (bền với nhiễu GPS), join vào `customer_zip_code_prefix` và `primary_seller_zip_code_prefix` (có sẵn ở `orders_joined.csv`, Story #4), tính khoảng cách haversine (km), rồi thêm cột này vào `orders_features_train.csv`/`orders_features_test.csv` đã có (Task #46/#47) — **không** đổi lại tập train/test đã split.

**Không rò rỉ dữ liệu**: zip code của khách và người bán chính đều đã xác định tại thời điểm đơn được duyệt thanh toán (giả định thời điểm dự đoán ở `docs/feature-list.md`), khớp cách `primary_seller` đã được chọn ở Story #4.

**Xử lý giá trị thiếu** (476/96.445 dòng train+test, 0.49%, do zip không khớp được `geolocation` hoặc đơn không có `primary_seller`): điền bằng **median khoảng cách của tập train**, áp dụng cho cả train và test — quyết định đã thống nhất với User để giữ nguyên đúng số dòng train/test hiện có (77.156/19.289), đảm bảo so sánh trước/sau công bằng ở notebook #24.

In [1]:
import numpy as np
import pandas as pd

geo = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
print("geolocation rows:", len(geo), " unique zip prefixes:", geo["geolocation_zip_code_prefix"].nunique())

geo_agg = (
    geo.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
    .median()
    .reset_index()
)
print("geo_agg rows (1/prefix):", len(geo_agg))
geo_agg.head()

geolocation rows: 1000163  unique zip prefixes: 19015
geo_agg rows (1/prefix): 19015


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,1001,-23.550381,-46.634027
1,1002,-23.548551,-46.635072
2,1003,-23.548977,-46.635313
3,1004,-23.549535,-46.634771
4,1005,-23.549612,-46.636532


## 1. Join toạ độ khách hàng + người bán chính, tính khoảng cách haversine

`orders_joined.csv` (Story #4) có `customer_zip_code_prefix` (không thiếu) và `primary_seller_zip_code_prefix` (thiếu 775 dòng — đơn không có `primary_seller`, đã ghi nhận từ Task #6).

In [2]:
def haversine_km(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


orders = pd.read_csv(
    "../data/processed/orders_joined.csv",
    usecols=["order_id", "customer_zip_code_prefix", "primary_seller_zip_code_prefix"],
)

distance = (
    orders
    .merge(geo_agg, left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="left")
    .rename(columns={"geolocation_lat": "customer_lat", "geolocation_lng": "customer_lng"})
    .drop(columns="geolocation_zip_code_prefix")
    .merge(geo_agg, left_on="primary_seller_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="left")
    .rename(columns={"geolocation_lat": "seller_lat", "geolocation_lng": "seller_lng"})
    .drop(columns="geolocation_zip_code_prefix")
)

distance["seller_customer_distance_km"] = haversine_km(
    distance["customer_lat"], distance["customer_lng"],
    distance["seller_lat"], distance["seller_lng"],
)

distance = distance[["order_id", "seller_customer_distance_km"]]
print("orders:", len(distance), " thieu khoang cach:", distance["seller_customer_distance_km"].isna().sum())
distance["seller_customer_distance_km"].describe()

orders: 99441  thieu khoang cach: 1264


count    98177.000000
mean       601.575862
std        595.528740
min          0.000000
25%        184.967779
50%        433.731480
75%        799.306422
max       8677.859564
Name: seller_customer_distance_km, dtype: float64

## 2. Thêm vào `orders_features_train.csv` / `orders_features_test.csv`, điền giá trị thiếu bằng median tập train

In [3]:
train_df = pd.read_csv("../data/processed/orders_features_train.csv", low_memory=False)
test_df = pd.read_csv("../data/processed/orders_features_test.csv", low_memory=False)

train_df = train_df.merge(distance, on="order_id", how="left")
test_df = test_df.merge(distance, on="order_id", how="left")

train_missing = train_df["seller_customer_distance_km"].isna().sum()
test_missing = test_df["seller_customer_distance_km"].isna().sum()
train_median = train_df["seller_customer_distance_km"].median()

print(f"Train: {len(train_df)} dong, thieu {train_missing} ({train_missing / len(train_df):.2%})")
print(f"Test:  {len(test_df)} dong, thieu {test_missing} ({test_missing / len(test_df):.2%})")
print(f"Median khoang cach (tap train, dung de dien): {train_median:.2f} km")

train_df["seller_customer_distance_km"] = train_df["seller_customer_distance_km"].fillna(train_median)
test_df["seller_customer_distance_km"] = test_df["seller_customer_distance_km"].fillna(train_median)

assert train_df["seller_customer_distance_km"].isna().sum() == 0
assert test_df["seller_customer_distance_km"].isna().sum() == 0
assert len(train_df) == 77156 and len(test_df) == 19289, "So dong train/test bi doi so voi Task #46!"

Train: 77156 dong, thieu 385 (0.50%)
Test:  19289 dong, thieu 91 (0.47%)
Median khoang cach (tap train, dung de dien): 434.70 km


## 3. Kiểm tra nhanh: khoảng cách có khác biệt giữa đơn trễ và đúng hạn không

In [4]:
train_df.groupby("is_delayed")["seller_customer_distance_km"].agg(["mean", "median", "count"])

,mean,median,count
is_delayed,,,
False,588.966603,429.777770,70895
True,739.431625,509.298049,6261


## 4. Lưu lại `orders_features_train.csv` / `orders_features_test.csv`

Ghi đè trực tiếp (thêm 1 cột `seller_customer_distance_km`, giữ nguyên toàn bộ cột và số dòng khác) — khớp cách các Task trước đã cập nhật 2 file này.

In [5]:
train_df.to_csv("../data/processed/orders_features_train.csv", index=False)
test_df.to_csv("../data/processed/orders_features_test.csv", index=False)
print("Da luu, so cot moi:", train_df.shape[1], "(truoc: 77)")

Da luu, so cot moi: 78 (truoc: 77)
